# ⚙️ Buổi 5/9 — Feature Engineering
### Tạo features mới từ domain knowledge để tăng hiệu quả model!

---

> **📍 Bạn đang ở đây trong lộ trình 9 buổi:**
> ```
> Buổi 1 ✅ Giới thiệu dự án & lộ trình
> Buổi 2 ✅ Setup môi trường & khám phá SalePrice
> Buổi 3 ✅ EDA — Phân tích missing, tương quan, outliers
> Buổi 4 ✅ Data Cleaning & Preprocessing
> Buổi 5 🔄 Feature Engineering                   ← BẠN ĐANG Ở ĐÂY
> Buổi 6 ⏳ Linear Regression & Regularization
> Buổi 7 ⏳ Tree-Based Models & Đánh Giá (RF, XGBoost, LightGBM)
> Buổi 8 ⏳ Submit Kaggle (tạo file & nộp bài)
> Buổi 9 ⏳ Quiz Tốt Nghiệp (20 câu ôn tập)
> ```

---

### 📋 Nội Dung Buổi 5

| # | Task | Nội dung | Kết quả |
|---|---|---|---|
| 1 | 📐 Area Features | TotalSF, TotalPorch, TotalBath, OverallScore | +4 features |
| 2 | 🕰️ Age Features | HouseAge, RemodAge, GarageAge, IsNew, HasRemod | +5 features |
| 3 | 📈 Transform Features | Squared, Sqrt, Log cho top predictors | +12 features |
| 4 | 🔍 Correlation Analysis | Top 20 features tương quan với target | Hiểu data hơn |
| 5 | 🗑️ Variance Filtering | Xoá features gần bằng 0 phương sai | Giảm nhiễu |
| 6 | ⚖️ Feature Scaling | Chuẩn hoá về cùng thang đo | Tối ưu cho Linear model |

---

### 🔗 Kết Nối Với Buổi 4

Ở Buổi 4, chúng ta đã **làm sạch** dữ liệu (0 missing, 2 outlier đã xoá, encoding xong).
Buổi 5 này chúng ta sẽ **sáng tạo thêm** — tạo features mới từ domain knowledge trước khi encode.

> ⚠️ **Lý do phải reload dữ liệu:** Chúng ta cần thêm features MỚI vào `all_data` **trước** bước One-Hot Encoding. Buổi 4 đã one-hot rồi, nên ở đây ta sẽ chạy lại pipeline ngắn gọn đến bước trước encoding, rồi thêm features mới, rồi mới encode + scale.

```
Pipeline Buổi 5:
load → fill → rm outlier → [TẠO FEATURE MỚI] → ordinal enc → variance filter → one-hot → scale → split
                                    ↑
                              Đây là phần mới!
```

In [ ]:
# ============================================================
# ♻️ SETUP — Import thư viện & tái tạo pipeline từ Buổi 4
# (chạy lại để có all_data TRƯỚC bước encoding)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

# ── Load dữ liệu gốc ──────────────────────────────────────
train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')

train_id = train_df['Id'].copy()
test_id  = test_df['Id'].copy()
y = np.log1p(train_df['SalePrice'])

train_df.drop(['Id', 'SalePrice'], axis=1, inplace=True)
test_df.drop(['Id'], axis=1, inplace=True)

ntrain = len(train_df)
ntest  = len(test_df)
all_data = pd.concat([train_df, test_df], axis=0, ignore_index=True)

# ── Fill missing (Chiến lược Buổi 4) ─────────────────────
none_cols = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu',
             'GarageType','GarageFinish','GarageQual','GarageCond',
             'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','MasVnrType']
zero_cols = ['GarageYrBlt','GarageArea','GarageCars','MasVnrArea',
             'BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath']
mode_cols = ['MSZoning','Utilities','Functional','Electrical',
             'KitchenQual','Exterior1st','Exterior2nd','SaleType']

for col in none_cols: all_data[col] = all_data[col].fillna('None')
for col in zero_cols: all_data[col] = all_data[col].fillna(0)
all_data['LotFrontage'] = (all_data.groupby('Neighborhood')['LotFrontage']
                           .transform(lambda x: x.fillna(x.median())))
for col in mode_cols: all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# ── Xoá outlier (2 nhà GrLivArea>4000 & SalePrice<300K) ──
outlier_mask = (all_data['GrLivArea'][:ntrain] > 4000) & (np.expm1(y) < 300_000)
outlier_idx  = outlier_mask[outlier_mask].index.tolist()
all_data.drop(index=outlier_idx, inplace=True)
all_data.reset_index(drop=True, inplace=True)
y.drop(index=outlier_idx, inplace=True)
y.reset_index(drop=True, inplace=True)
ntrain = ntrain - len(outlier_idx)   # 1458

print("✅ Pipeline Buổi 4 đã chạy lại thành công!")
print(f"   all_data  : {all_data.shape}  (trước encoding)")
print(f"   y         : {y.shape}  (log SalePrice)")
print(f"   ntrain    : {ntrain}  |  ntest : {ntest}")
print(f"   Missing   : {all_data.isnull().sum().sum()} ô")
print()
print("⏭️ Bước tiếp theo: Thêm FEATURES MỚI trước khi encode...")

**Expected Output:**

![img5-1](images/img_buoi5/img5-1.png)    

---

## ⚙️ Task 1 — Tạo Features Mới (Feature Creation)

---

### 🤔 Feature Engineering Là Gì? Tại Sao Quan Trọng?

**Feature Engineering** là nghệ thuật tạo ra **biến mới** từ dữ liệu hiện có, dựa trên hiểu biết về lĩnh vực (domain knowledge) để giúp mô hình "nhìn thấy" những pattern mà dữ liệu thô không thể hiện rõ.

```
Dữ liệu gốc:    1stFlrSF=1200,  2ndFlrSF=800,  TotalBsmtSF=1000
                                        ↓ Feature Engineering
Feature mới:    TotalSF = 1200 + 800 + 1000 = 3000 sqft

Tại sao tốt hơn?
→ Model biết "tổng diện tích sống" trực tiếp thay vì phải tự cộng 3 cột!
→ Hệ số correlation với SalePrice thường cao hơn từng phần riêng lẻ
```

### 🌍 Các Loại Feature Engineering Phổ Biến

| Loại | Ví dụ trong dự án này | Dùng khi nào |
|---|---|---|
| **Tổng hợp (Aggregation)** | TotalSF, TotalBath | Nhiều cột cùng loại, muốn gộp lại |
| **Tính tuổi/thời gian** | HouseAge, RemodAge | Dữ liệu có năm tháng |
| **Binary flag** | IsNew, HasGarage | Phân biệt có/không |
| **Nhân chéo (Interaction)** | OverallQual × OverallCond | 2 features ảnh hưởng nhau |
| **Biến đổi toán học** | log(GrLivArea), GrLivArea² | Phân phối lệch hoặc quan hệ phi tuyến |
| **Tỉ lệ (Ratio)** | LivAreaPerRoom = GrLivArea/TotRmsAbvGrd | Normalize theo quy mô |

> 💡 **Bí quyết:** Đặt mình vào vị trí người mua nhà — họ quan tâm đến gì khi quyết định giá? Diện tích tổng, tuổi nhà, số phòng tắm,... Đó chính là nguồn cảm hứng để tạo features!

In [ ]:
# ============================================================
# 📐 TASK 1.1 — AREA FEATURES (Features diện tích tổng hợp)
# ============================================================

shape_before_fe = all_data.shape
print(f"📊 all_data trước khi thêm features: {shape_before_fe}")
print()

# ── Feature 1: Tổng diện tích sống ────────────────────────
# Người mua nhà quan tâm tổng số sqft, không phải từng tầng riêng lẻ
all_data['TotalSF'] = (all_data['TotalBsmtSF'] +
                       all_data['1stFlrSF']    +
                       all_data['2ndFlrSF'])

# ── Feature 2: Tổng diện tích hiên/sân ───────────────────
# Gộp tất cả loại porch → giá trị tổng thể tốt hơn
all_data['TotalPorchSF'] = (all_data['OpenPorchSF']    +
                             all_data['3SsnPorch']      +
                             all_data['EnclosedPorch']  +
                             all_data['ScreenPorch']    +
                             all_data['WoodDeckSF'])

# ── Feature 3: Tổng số phòng tắm ──────────────────────────
# Phòng tắm đầy tính 1.0, phòng tắm nửa (chỉ có sink+toilet) tính 0.5
all_data['TotalBath'] = (all_data['FullBath']      +
                         0.5 * all_data['HalfBath'] +
                         all_data['BsmtFullBath']  +
                         0.5 * all_data['BsmtHalfBath'])

# ── Feature 4: Điểm tổng hợp Chất lượng × Tình trạng ────
# Interaction feature: nhà chất lượng CAO + tình trạng TỐT → giá cao nhất
all_data['OverallScore'] = all_data['OverallQual'] * all_data['OverallCond']

# ── Feature 5: Có garage không? (binary) ──────────────────
all_data['HasGarage'] = (all_data['GarageArea'] > 0).astype(int)

# ── Feature 6: Có tầng hầm không? (binary) ────────────────
all_data['HasBsmt'] = (all_data['TotalBsmtSF'] > 0).astype(int)

# ── Kiểm tra tương quan các feature mới với target ────────
train_part = all_data[:ntrain].copy()
new_area_features = ['TotalSF', 'TotalPorchSF', 'TotalBath', 'OverallScore', 'HasGarage', 'HasBsmt']
corr_new = train_part[new_area_features].corrwith(y).abs().sort_values(ascending=False)

print("✅ Đã tạo Area Features:")
print()
for feat in new_area_features:
    print(f"   {feat:15s} | {all_data[feat].mean():.1f} (mean) | corr={corr_new.get(feat, 0):.3f}")

print()
print(f"📊 Số cột hiện tại: {all_data.shape[1]} (tăng {all_data.shape[1] - shape_before_fe[1]} cột)")

**Expected Output:**

![img5-2](images/img_buoi5/img5-2.png)

---

### 🔬 Giải Thích — Area Features

---

```python
all_data['TotalBath'] = all_data['FullBath'] + 0.5 * all_data['HalfBath'] + ...
```

- **`0.5 * HalfBath`** — phòng tắm nửa (chỉ có toilet + lavabo, không có bồn tắm/vòi sen) giá trị chỉ bằng một nửa phòng đầy đủ. Đây là **domain knowledge từ thị trường bất động sản Mỹ**
- Cộng cả tầng hầm (`BsmtFullBath`, `BsmtHalfBath`) vì người mua vẫn tính là tiện ích

```python
all_data['OverallScore'] = all_data['OverallQual'] * all_data['OverallCond']
```

- **Interaction feature:** `OverallQual` (chất lượng vật liệu 1-10) × `OverallCond` (tình trạng tổng thể 1-10)
- Nhà điểm 9×9=81 và nhà 5×5=25 — model hiểu rõ khoảng cách hơn

---

### 🌐 Mở Rộng — Các Cách Tạo Area Feature Khác

**Cách khác cho dự án khác:**

| Tình huống | Feature mới | Công thức |
|---|---|---|
| E-commerce | Giá trung bình/đơn | `TotalRevenue / NumOrders` |
| HR Analytics | Năng suất nhân viên | `SalesAmount / WorkingHours` |
| Credit scoring | Tỷ lệ nợ | `DebtAmount / MonthlyIncome` |
| Nhà đất | Giá/sqft | `SalePrice / TotalSF` |

```python
# Ví dụ: ratio feature (dùng thêm trong bài toán này)
all_data['LivAreaRatio'] = all_data['GrLivArea'] / all_data['TotRmsAbvGrd']
# → "Diện tích trung bình mỗi phòng" — nhà rộng ít phòng khác nhà hẹp nhiều phòng
```

**🎯 Mục đích sau cùng của Task 1.1:** `TotalSF` thường có **correlation >0.80** với SalePrice — cao hơn nhiều so với `1stFlrSF` riêng lẻ (~0.60). Feature tổng hợp giúp model "nhìn thấy bức tranh toàn cảnh".

In [ ]:
# ============================================================
# 🕰️ TASK 1.2 — AGE FEATURES (Features liên quan đến thời gian)
# ============================================================

# Dùng YrSold (năm bán thực tế) thay vì năm cố định
# → "Tuổi nhà khi bán" có ý nghĩa hơn "tuổi nhà tính đến 2024"
# Ví dụ: nhà xây 1990, bán 2008 → HouseAge=18 năm khi bán

# ── Feature 1: Tuổi nhà tại thời điểm bán ─────────────────
all_data['HouseAge']  = all_data['YrSold'] - all_data['YearBuilt']

# ── Feature 2: Năm kể từ lần sửa chữa gần nhất ───────────
all_data['RemodAge']  = all_data['YrSold'] - all_data['YearRemodAdd']

# ── Feature 3: Tuổi garage (0 nếu không có garage) ───────
all_data['GarageAge'] = all_data['YrSold'] - all_data['GarageYrBlt']
all_data['GarageAge'] = all_data['GarageAge'].clip(lower=0)   # xử lý trường hợp âm

# ── Feature 4: Nhà mới? (xây sau 2000) ────────────────────
all_data['IsNew']     = (all_data['YearBuilt'] >= 2000).astype(int)

# ── Feature 5: Đã được sửa chữa? ─────────────────────────
# Nếu YearRemodAdd == YearBuilt → chưa bao giờ sửa
all_data['HasRemod']  = (all_data['YearRemodAdd'] != all_data['YearBuilt']).astype(int)

# ── Feature 6: Nhà được bán trong năm xây (nhà mới tinh) ─
all_data['SoldAsNew'] = (all_data['YrSold'] == all_data['YearBuilt']).astype(int)

# ── Kiểm tra phân phối và tương quan ──────────────────────
age_features = ['HouseAge', 'RemodAge', 'GarageAge', 'IsNew', 'HasRemod', 'SoldAsNew']
train_part   = all_data[:ntrain].copy()
corr_age     = train_part[age_features].corrwith(y).sort_values()

print("✅ Age Features đã tạo:")
print()
print(f"   {'Feature':<15} {'Mean':>8} {'Min':>6} {'Max':>6}  {'Corr(y)':>9}")
print("   " + "─" * 50)
for feat in age_features:
    col  = all_data[feat]
    corr = corr_age.get(feat, 0)
    print(f"   {feat:<15} {col.mean():>8.1f} {col.min():>6.0f} {col.max():>6.0f}  {corr:>+9.3f}")

print()
print(f"📊 Số cột hiện tại: {all_data.shape[1]}")

**Expected Output:**

![img5-3](images/img_buoi5/img5-3.png)

---

### 🔬 Giải Thích — Age Features

---

```python
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
```

**Tại sao dùng `YrSold` thay vì năm hiện tại (`2024`)?**

| Cách | Công thức | Ý nghĩa |
|---|---|---|
| ✅ Dùng `YrSold` | `2008 - 1990 = 18` | Tuổi nhà **tại thời điểm bán** — phản ánh đúng giá trị lúc giao dịch |
| ❌ Dùng `2024` | `2024 - 1990 = 34` | Tuổi nhà tính đến hôm nay — không liên quan đến giá bán lịch sử |

Dataset này ghi nhận giao dịch từ 2006–2010. Người mua nhà năm 2008 không biết giá năm 2024!

```python
all_data['GarageAge'] = all_data['GarageAge'].clip(lower=0)
```

- **`.clip(lower=0)`** — giới hạn giá trị tối thiểu là 0, tránh giá trị âm (do GarageYrBlt=0 đã fill ở Buổi 4)
- Tương đương nhưng ngắn gọn hơn: `replace` hay `where` (dùng khi logic phức tạp hơn)

```python
all_data['HasRemod'] = (all_data['YearRemodAdd'] != all_data['YearBuilt']).astype(int)
```

- **Phép so sánh `!=`** tạo ra Series True/False
- **`.astype(int)`** chuyển True→1, False→0 để model xử lý được

---

### 🌐 Mở Rộng — Các Bài Toán Khác Cần Age Features

```python
# E-commerce: "Thời gian kể từ lần mua cuối"
df['DaysSinceLastPurchase'] = (today - df['LastPurchaseDate']).dt.days

# Churn prediction: "Tuổi tài khoản"
df['AccountAge'] = (today - df['SignupDate']).dt.days / 365

# Nhân sự: "Số năm kinh nghiệm"
df['Experience'] = df['CurrentYear'] - df['HireYear']
```

**🎯 Mục đích sau cùng của Task 1.2:** `HouseAge` thường có tương quan **âm** với SalePrice (nhà càng cũ giá càng thấp). `IsNew=1` thường đẩy giá cao hơn. Đây là những insights quan trọng mà model sẽ "học" được.

In [ ]:
# ============================================================
# 📈 TASK 1.3 — POLYNOMIAL & TRANSFORM FEATURES
# (Biến đổi toán học để nắm bắt quan hệ phi tuyến)
# ============================================================

# Top features tương quan cao nhất — dùng cho polynomial
top_features = ['OverallQual', 'GrLivArea', 'TotalSF']

for feat in top_features:
    all_data[f'{feat}_sq']   = all_data[feat] ** 2          # bình phương
    all_data[f'{feat}_sqrt'] = np.sqrt(all_data[feat])       # căn bậc hai

# Log transform cho các features có phân phối lệch (skewed)
# log1p an toàn hơn log vì xử lý được giá trị 0
skewed_features = ['GrLivArea', 'TotalSF', 'LotArea', '1stFlrSF', 'TotalPorchSF']
for feat in skewed_features:
    if feat in all_data.columns:
        skewness = all_data[feat].skew()
        all_data[f'{feat}_log'] = np.log1p(all_data[feat])
        print(f"   {feat:<18} skew={skewness:+.2f} → log transform")

print()
# ── Trực quan hoá tác động của transform ──────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Tác Động của Transform Features\n(Phân phối trước và sau biến đổi)', fontsize=13)

pairs = [
    ('GrLivArea',     'GrLivArea_log',  'Log Transform'),
    ('TotalSF',       'TotalSF_sq',     'Squared'),
    ('OverallQual',   'OverallQual_sq', 'Squared (Ordinal)'),
]

train_part = all_data[:ntrain].copy()
for idx, (orig, trans, label) in enumerate(pairs):
    # Trước
    ax1 = axes[0, idx]
    ax1.hist(train_part[orig].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax1.set_title(f'{orig} (gốc)')
    ax1.set_xlabel(f'skew = {train_part[orig].skew():.2f}')

    # Sau
    ax2 = axes[1, idx]
    ax2.hist(train_part[trans].dropna(), bins=40, color='coral', edgecolor='white', alpha=0.8)
    ax2.set_title(f'{trans} ({label})')
    ax2.set_xlabel(f'skew = {train_part[trans].skew():.2f}')

plt.tight_layout()
plt.show()

**Expected Output:**

![img5-4](images/img_buoi5/img5-4.png)
![img5-5](images/img_buoi5/img5-5.png)

---

### 🔬 Giải Thích — Polynomial & Transform Features

---

**Tại sao cần biến đổi toán học?**

Linear Regression giả định quan hệ **tuyến tính** giữa feature và target:
```
SalePrice = a × GrLivArea + b   ← giả định tuyến tính

Thực tế:
   Nhà 1000→2000 sqft: giá tăng mạnh (nhà quá nhỏ → nhà thoải mái)
   Nhà 3000→4000 sqft: giá tăng ít hơn (đã đủ rộng, thêm diện tích ít giá trị hơn)
   → Quan hệ PHI TUYẾN → cần sqrt hoặc log!
```

---

### 🌐 So Sánh Các Phép Biến Đổi — Chọn Cái Nào?

| Biến đổi | Công thức | Dùng khi nào | Ví dụ |
|---|---|---|---|
| **Log** `log1p(x)` | $\ln(x+1)$ | Phân phối lệch phải, giá trị dương | Thu nhập, diện tích, giá |
| **Sqrt** `√x` | $\sqrt{x}$ | Lệch nhẹ, muốn nén nhẹ | Số lượt click, số đánh giá |
| **Squared** `x²` | $x^2$ | Quan hệ bậc 2 với target | Khoảng cách, nhiệt độ |
| **Box-Cox** | $\frac{x^\lambda-1}{\lambda}$ | Tự động tìm $\lambda$ tốt nhất | Khi không chắc chọn gì |
| **Reciprocal** `1/x` | $\frac{1}{x}$ | Quan hệ nghịch đảo | Tốc độ, mật độ |

```python
# Box-Cox (dùng khi muốn tự động hóa việc chọn transform):
from scipy import stats
all_data['GrLivArea_boxcox'], fitted_lambda = stats.boxcox(all_data['GrLivArea'] + 1)
# scipy tự tìm lambda tối ưu → tốt nhất về mặt thống kê, nhưng khó giải thích

# Yeo-Johnson (giống Box-Cox nhưng xử lý được giá trị âm):
from sklearn.preprocessing import PowerTransformer
pt = PowerTransformer(method='yeo-johnson')
# → Thường dùng trong AutoML pipeline
```

**💡 Nguyên tắc chọn:**
- Phân phối lệch phải nhiều (skew > 1.5) → **log1p**
- Skew vừa phải → **sqrt**
- Muốn tự động → **Box-Cox / Yeo-Johnson**

**🎯 Mục đích sau cùng của Task 1.3:** Bằng cách thêm `GrLivArea_log`, chúng ta giúp Linear Regression "nhìn thấy" quan hệ phi tuyến mà không cần đổi sang model phức tạp hơn.

---

## 🔍 Task 2 — Lựa Chọn Features Quan Trọng (Feature Selection)

---

### 🤔 Tại Sao Cần Chọn Features?

Sau khi tạo thêm features, chúng ta có thể có **200+ cột**. Không phải cột nào cũng hữu ích:

```
❌ Vấn đề khi có quá nhiều features (Curse of Dimensionality):
   → Một số features nhiễu (noise) có thể làm model overfit
   → Training chậm hơn
   → Một số features gần như hằng số (variance ≈ 0) → vô nghĩa
   → Features tương quan cao với nhau → multicollinearity
```

### 📋 Các Phương Pháp Feature Selection

| Phương pháp | Cách hoạt động | Ưu điểm | Nhược điểm |
|---|---|---|---|
| **Correlation** | Tương quan với target | Đơn giản, nhanh | Bỏ qua feature kết hợp |
| **VarianceThreshold** | Loại variance ≈ 0 | Không cần target | Ngưỡng khó chọn |
| **RFE** (Recursive FE) | Đệ quy loại feature xấu | Mạnh, tổng quát | Chậm với dataset lớn |
| **L1/Lasso** | Coeff = 0 → loại | Embedded, hiệu quả | Cần tune hyperparameter |
| **Mutual Information** | Thông tin chia sẻ | Nắm bắt phi tuyến | Chậm hơn correlation |

> 💡 **Trong thực tế:** Người ta thường kết hợp nhiều phương pháp, bắt đầu từ đơn giản nhất (correlation + variance) rồi dùng model-based nếu cần.

In [ ]:
# ============================================================
# 🔗 TASK 2.0 — ORDINAL ENCODING (trước khi split & phân tích)
# (cần encode ordinal TRƯỚC khi tính correlation vì cần số)
# ============================================================

ordinal_mappings = {
    'ExterQual'   : ['None','Po','Fa','TA','Gd','Ex'],
    'ExterCond'   : ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtQual'    : ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtCond'    : ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtExposure': ['None','No','Mn','Av','Gd'],
    'BsmtFinType1': ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'BsmtFinType2': ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'HeatingQC'   : ['None','Po','Fa','TA','Gd','Ex'],
    'KitchenQual' : ['None','Po','Fa','TA','Gd','Ex'],
    'FireplaceQu' : ['None','Po','Fa','TA','Gd','Ex'],
    'GarageFinish': ['None','Unf','RFn','Fin'],
    'GarageQual'  : ['None','Po','Fa','TA','Gd','Ex'],
    'GarageCond'  : ['None','Po','Fa','TA','Gd','Ex'],
    'PoolQC'      : ['None','Fa','TA','Gd','Ex'],
    'Fence'       : ['None','MnWw','GdWo','MnPrv','GdPrv'],
    'Functional'  : ['Sal','Sev','Maj2','Maj1','Mod','Min2','Min1','Typ'],
    'LandSlope'   : ['Sev','Mod','Gtl'],
    'LotShape'    : ['IR3','IR2','IR1','Reg'],
    'PavedDrive'  : ['N','P','Y'],
}

for col, order in ordinal_mappings.items():
    mapping_dict = {val: i for i, val in enumerate(order)}
    all_data[col] = all_data[col].map(mapping_dict).fillna(0).astype(int)

print(f"✅ Ordinal Encoding xong: {len(ordinal_mappings)} cột")
print(f"   all_data shape: {all_data.shape}")

**Expected Output:**

![img5-6](images/img_buoi5/img5-6.png)

In [ ]:
# ============================================================
# 📊 TASK 2.1 — CORRELATION ANALYSIS
# (Phân tích tương quan features với target)
# ============================================================

# One-hot encode để tính correlation (cần toàn bộ số)
cat_cols  = all_data.select_dtypes(include=['object']).columns.tolist()
all_data_enc = pd.get_dummies(all_data, columns=cat_cols, drop_first=True)

# Split lại thành train/test
X_train_raw = all_data_enc[:ntrain].copy()
X_test_raw  = all_data_enc[ntrain:].copy()
X_test_raw.reset_index(drop=True, inplace=True)

print(f"📐 Sau One-Hot: {X_train_raw.shape[1]} features tổng cộng")
print()

# Tính correlation với target
correlations = X_train_raw.corrwith(y).abs().sort_values(ascending=False)

# Hiển thị top 25
print("📊 Top 25 Features có tương quan cao nhất với log(SalePrice):")
print()
print(f"   {'#':>3} {'Feature':<30} {'|Corr|':>8}  {'Bar':}")
print("   " + "─" * 60)
for rank, (feat, corr) in enumerate(correlations.head(25).items(), 1):
    filled  = int(corr * 25)
    empty   = 25 - filled
    bar = "━" * filled + "╌" * empty
    print(f"   {rank:>3}. {feat:<30} {corr:.4f}  [{bar}]")

# ── Vẽ bar chart top 20 ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 9))
top20 = correlations.head(20)
colors = ['#2196F3' if i < 5 else '#64B5F6' if i < 10 else '#BBDEFB' for i in range(20)]
top20[::-1].plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='white')
ax.set_xlabel('Absolute Correlation với log(SalePrice)', fontsize=11)
ax.set_title('Top 20 Features — Tương Quan Với Target\n(Buổi 5 | bao gồm features mới)', fontsize=12)
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='Ngưỡng 0.5')
ax.legend()
for i, v in enumerate(top20[::-1].values):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

**Expected Output:**

![img5-7](images/img_buoi5/img5-7.png)
![img5-8](images/img_buoi5/img5-8.png)

---

### 🔬 Giải Thích — Correlation Analysis

---

```python
correlations = X_train_raw.corrwith(y).abs().sort_values(ascending=False)
```

| Lệnh | Ý nghĩa |
|---|---|
| `.corrwith(y)` | Tính Pearson correlation giữa **mỗi cột** của DataFrame với Series `y` |
| `.abs()` | Lấy giá trị tuyệt đối — quan hệ âm mạnh cũng quan trọng không kém dương |
| `.sort_values(ascending=False)` | Sắp xếp từ cao đến thấp |

**Diễn giải kết quả correlation:**

| Khoảng | Ý nghĩa | Quyết định |
|---|---|---|
| > 0.7 | Tương quan **rất mạnh** | Chắc chắn giữ |
| 0.4 – 0.7 | Tương quan **trung bình** | Nên giữ |
| 0.1 – 0.4 | Tương quan **yếu** | Xem xét thêm |
| < 0.1 | Gần như **không tương quan** | Cân nhắc loại |

---

### 🌐 Mở Rộng — Correlation Không Phải "Chân Lý"

```
Ví dụ: PoolArea có correlation thấp với SalePrice (~0.06)
Nhưng nếu kết hợp với PoolQC → nhà có bể bơi chất lượng cao → giá cao vọt!
→ Feature interaction mà single-feature correlation KHÔNG nắm bắt được

→ Giải pháp: dùng Mutual Information (nắm bắt quan hệ phi tuyến và tương tác)
```

```python
# Mutual Information (sklearn) — đo "thông tin chia sẻ" phi tuyến:
from sklearn.feature_selection import mutual_info_regression
mi_scores = mutual_info_regression(X_train_raw.fillna(0), y, random_state=42)
mi_series = pd.Series(mi_scores, index=X_train_raw.columns).sort_values(ascending=False)
```

**🎯 Mục đích sau cùng:** Hiểu được feature nào THỰC SỰ quan trọng để tập trung. Kết quả bất ngờ: `TotalSF` (feature mới tạo) thường đứng top 3 — xác nhận Feature Engineering có giá trị!

In [ ]:
# ============================================================
# 🗑️ TASK 2.2 — REMOVE LOW VARIANCE FEATURES
# (Xoá các features gần như hằng số — vô ích với model)
# ============================================================
from sklearn.feature_selection import VarianceThreshold

print("🔍 Phân tích Variance của các features:")
print()

# Tính variance của mỗi cột trong X_train
variances = X_train_raw.var().sort_values()
very_low_var = variances[variances < 0.01]
print(f"   Features có variance < 0.01 (gần hằng số): {len(very_low_var)}")
if len(very_low_var) > 0:
    print("   Danh sách:")
    for feat, var in very_low_var.items():
        val_counts = X_train_raw[feat].value_counts()
        dominant = val_counts.iloc[0] / len(X_train_raw) * 100
        print(f"      {feat:<35} var={var:.5f} | dominant_val={dominant:.1f}%")

print()

# Áp dụng VarianceThreshold
selector = VarianceThreshold(threshold=0.01)
selector.fit(X_train_raw)

selected_mask    = selector.get_support()
removed_features = X_train_raw.columns[~selected_mask].tolist()
kept_features    = X_train_raw.columns[selected_mask].tolist()

print(f"📊 Kết quả Variance Filtering (threshold=0.01):")
print(f"   Features ban đầu : {X_train_raw.shape[1]}")
print(f"   Features bị loại : {len(removed_features)}")
print(f"   Features giữ lại : {len(kept_features)}")
print()

# Áp dụng selection
X_train_sel = X_train_raw.loc[:, selected_mask].copy()
X_test_sel  = X_test_raw.loc[:, selected_mask].copy()

print(f"   X_train : {X_train_sel.shape}")
print(f"   X_test  : {X_test_sel.shape}")

**Expected Output:**

![img5-9](images/img_buoi5/img5-9.png)

---

### 🔬 Giải Thích — Variance Filtering

---

```python
selector = VarianceThreshold(threshold=0.01)
selector.fit(X_train_raw)
selected_mask = selector.get_support()
```

| Lệnh | Ý nghĩa |
|---|---|
| `VarianceThreshold(threshold=0.01)` | Tạo bộ lọc: loại feature có variance < 0.01 |
| `.fit(X_train_raw)` | Tính variance của từng cột trên **train** (KHÔNG dùng test) |
| `.get_support()` | Trả về mảng True/False — True = feature được giữ |

> ⚠️ **Quan trọng:** `fit()` chỉ trên **train**, rồi apply `transform` cho cả train lẫn test — tránh **data leakage** (thông tin test rò vào quá trình huấn luyện)

---

### 🌐 Mở Rộng — Các Ngưỡng Variance Khác Nhau

```python
# Ngưỡng thấp (conservative) — chỉ loại feature toàn một giá trị
selector = VarianceThreshold(threshold=0.0)      # loại "zero variance"

# Ngưỡng vừa (recommended)
selector = VarianceThreshold(threshold=0.01)     # ← đang dùng

# Ngưỡng cao (aggressive) — cẩn thận có thể loại feature quan trọng
selector = VarianceThreshold(threshold=0.05)

# Ví dụ thực tế ngoài bất động sản:
# Dataset tín dụng: cột "HasDefaulted" (99% = 0, 1% = 1)
# → Variance rất thấp NHƯNG cực kỳ quan trọng!
# → Đừng mù quáng dùng threshold cao
```

**⚡ Kết hợp với Correlation để ra quyết định tốt hơn:**

```python
# Loại feature vừa variance thấp VÀ correlation thấp
low_var    = set(X_train_raw.columns[~selector.get_support()])
low_corr   = set(correlations[correlations < 0.05].index)
to_remove  = low_var | low_corr    # hợp 2 tập
```

**🎯 Mục đích sau cùng:** Giảm "tiếng ồn" (noise), training nhanh hơn, tránh overfitting từ features vô nghĩa.

In [ ]:
# ============================================================
# ⚖️ TASK 2.3 — FEATURE SCALING (Chuẩn Hoá Thang Đo)
# ============================================================
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

print("📊 Thống kê TRƯỚC khi scale (một số features tiêu biểu):")
sample_feats = ['TotalSF', 'GrLivArea', 'LotArea', 'OverallQual']
available = [f for f in sample_feats if f in X_train_sel.columns]
print(X_train_sel[available].describe().loc[['mean','std','min','max']].round(1).to_string())
print()

# ── Áp dụng StandardScaler ────────────────────────────────
# fit trên train → transform cả train lẫn test
scaler = StandardScaler()
X_train_scaled_arr = scaler.fit_transform(X_train_sel)
X_test_scaled_arr  = scaler.transform(X_test_sel)

# Chuyển lại thành DataFrame (giữ tên cột)
X_train = pd.DataFrame(X_train_scaled_arr,
                        columns=X_train_sel.columns,
                        index=X_train_sel.index)
X_test  = pd.DataFrame(X_test_scaled_arr,
                        columns=X_test_sel.columns,
                        index=X_test_sel.index)

print("📊 Thống kê SAU khi scale (StandardScaler):")
print(X_train[available].describe().loc[['mean','std','min','max']].round(3).to_string())
print()

print("✅ Feature Scaling hoàn tất!")
print(f"   X_train : {X_train.shape}")
print(f"   X_test  : {X_test.shape}")
print()
print("🎯 Kiểm tra: mean ≈ 0, std ≈ 1 cho mỗi feature (StandardScaler đảm bảo điều này)")

**Expected Output:**

![img5-10](images/img_buoi5/img5-10.png)

---

### 🔬 Giải Thích — Feature Scaling

---

**Tại sao cần scaling?**

```
Ví dụ không scale:
   LotArea  : 65,000 sqft  (phạm vi: 1,300 – 215,000)
   OverallQual: 7           (phạm vi: 1 – 10)
   
Khi Ridge/Lasso tính penalty: LotArea bị phạt NHIỀU HƠN chỉ vì số lớn hơn!
→ Model bị lệch, không công bằng
→ Sau scale: cả hai đều có mean=0, std=1 → bình đẳng
```

```python
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)   # fit + transform
X_test_scaled  = scaler.transform(X_test_sel)         # chỉ transform (KHÔNG fit lại!)
```

> ⚠️ **Lỗi thường gặp:** Gọi `fit_transform` trên cả test → **data leakage!** Test set "nhìn" thấy thống kê train. Luôn chỉ `transform()` trên test.

---

### 🌐 So Sánh 3 Scaler Phổ Biến

| Scaler | Công thức | Ưu điểm | Dùng khi nào |
|---|---|---|---|
| **StandardScaler** | $z = \frac{x - \mu}{\sigma}$ | Phổ biến, mạnh với phân phối chuẩn | Regression, SVM, Neural Net |
| **RobustScaler** | $z = \frac{x - median}{IQR}$ | Kháng outlier tốt | Data còn outlier nhiều |
| **MinMaxScaler** | $z = \frac{x - min}{max - min}$ | Giới hạn [0,1] | Neural Net, KNN |

```python
# Ví dụ dùng RobustScaler (tốt hơn khi data có outliers):
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()          # dùng median & IQR thay vì mean & std
X_train_scaled = scaler.fit_transform(X_train_sel)

# MinMaxScaler (ép về [0,1]) — phổ biến cho deep learning:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
```

**📌 Quyết định trong dự án này:** Dùng `StandardScaler` vì đã xoá outlier ở Buổi 4 và sẽ dùng Ridge/Lasso ở Buổi 6. Tree-based models (RF, XGBoost ở Buổi 7) KHÔNG cần scale!

**🎯 Mục đích sau cùng của Task 2:** `X_train` và `X_test` bây giờ là **ma trận số sạch, chuẩn hoá** — đầu vào lý tưởng cho mọi ML algorithm. Buổi 6 sẽ train model trực tiếp trên đây!

---

## 📝 Bài Tập Buổi 5

> **Mục tiêu:** Ôn lại những gì đã học — tạo binary feature, xem feature quan trọng nhất, kiểm tra scaler.

---

### Bài Tập 1 — Tạo Feature `HasPool`

Trong buổi học ta đã tạo các binary feature như `HasGarage`, `HasBsmt`. Hãy tự làm tương tự với `PoolArea`:

1. Tạo cột `HasPool = (PoolArea > 0).astype(int)` trong `all_data`
2. In ra số nhà có bể bơi và không có bể bơi (`value_counts()`)
3. Tính correlation của `HasPool` với `y` trên phần train

> 💡 Gợi ý: `(all_data['PoolArea'] > 0).astype(int)` | `all_data[:ntrain]['HasPool'].corr(y)`

In [ ]:
# ============================================================
# BÀI TẬP 1 — Tạo feature HasPool
# ============================================================

# TODO 1: Tạo cột HasPool
all_data['HasPool'] = (all_data[???] > 0).astype(???)   # TODO: tên cột và kiểu int

# TODO 2: Đếm số nhà có / không có bể bơi
print("Số nhà theo HasPool:")
print(all_data['HasPool'].???)   # TODO: hàm đếm từng giá trị

# TODO 3: Tính correlation với y
corr_haspool = all_data[:ntrain]['HasPool'].???(y)   # TODO: hàm tính correlation
print(f"\nCorrelation của HasPool với SalePrice (log): {corr_haspool:.4f}")

**Expected Output:**

![img5-11](images/img_buoi5/img5-11.png)

<details>
<summary>👆 Click để xem đáp án Bài Tập 1</summary>

```python
all_data['HasPool'] = (all_data['PoolArea'] > 0).astype(int)

print(all_data['HasPool'].value_counts())
# 0    2909   ← hầu hết không có bể bơi
# 1      10   ← chỉ 10 nhà có bể bơi

corr_haspool = all_data[:ntrain]['HasPool'].corr(y)
# → Khoảng 0.07 — khá thấp
```

**Nhận xét:** Chỉ có ~10 nhà có bể bơi trong toàn bộ dataset → feature rất sparse → correlation thấp. Đây là lý do nhiều binary feature không mạnh bằng numerical feature.

</details>

---

### Bài Tập 2 — Xem Top 10 Features Tương Quan Nhất

Trong buổi học, biến `correlations` đã được tính (correlation của tất cả features với `y`). Hãy in ra top 10:

1. In ra 10 features có correlation **cao nhất** (dương)
2. In ra 5 features có correlation **thấp nhất** (âm mạnh nhất)

> 💡 Gợi ý: `correlations.sort_values(ascending=False).head(10)` | `.tail(5)`

In [ ]:
# ============================================================
# BÀI TẬP 2 — Xem top features tương quan nhất
# ============================================================

# TODO 1: Top 10 features correlation cao nhất
print("📈 Top 10 features tương quan CAO nhất với SalePrice:")
print(correlations.sort_values(ascending=???).head(???))   # TODO: False và 10

print()

# TODO 2: Top 5 features correlation âm mạnh nhất
print("📉 Top 5 features tương quan ÂM mạnh nhất:")
print(correlations.sort_values(ascending=???).head(???))   # TODO: True và 5

**Expected Output:**

![img5-12](images/img_buoi5/img5-12.png)

<details>
<summary>👆 Click để xem đáp án Bài Tập 2</summary>

```python
print(correlations.sort_values(ascending=False).head(10))
# TotalSF_log      0.82...
# TotalSF          0.81...
# OverallQual      0.79...
# ...

print(correlations.sort_values(ascending=True).head(5))
# HouseAge    -0.55...  ← nhà càng cũ giá càng thấp
# ...
```

**Nhận xét:** `TotalSF_log` (tổng diện tích sau log transform) thường đứng đầu — feature engineering đã tạo ra feature mạnh hơn các cột gốc.

</details>

---

### Bài Tập 3 — Kiểm Tra StandardScaler Đã Hoạt Động Đúng Chưa

Sau khi scale xong, `X_train` nên có **mean ≈ 0** và **std ≈ 1** cho mỗi cột. Hãy kiểm tra:

1. Tính mean và std của toàn bộ `X_train` (trên tất cả cột)
2. In ra mean và std trung bình để xác nhận StandardScaler đã hoạt động đúng

> 💡 Gợi ý: `X_train.mean().mean()` để lấy mean của tất cả mean | `X_train.std().mean()` để lấy std trung bình

In [ ]:
# ============================================================
# BÀI TẬP 3 — Kiểm tra StandardScaler đã hoạt động đúng
# ============================================================

# TODO 1: Tính mean của tất cả cột trong X_train
all_means = X_train.???()   # TODO: hàm tính mean theo cột
print(f"Mean trung bình toàn X_train: {all_means.mean():.6f}")   # ← kỳ vọng ≈ 0

# TODO 2: Tính std của tất cả cột trong X_train
all_stds = X_train.???()    # TODO: hàm tính std theo cột
print(f"Std trung bình toàn X_train : {all_stds.mean():.6f}")    # ← kỳ vọng ≈ 1

# TODO 3: Nhận xét
print("\nStandardScaler hoạt động đúng không? (mean ≈ 0 và std ≈ 1)")
print(f"  Mean ≈ 0? {'✅ Đúng' if abs(all_means.mean()) < 0.01 else '❌ Không đúng'}")
print(f"  Std  ≈ 1? {'✅ Đúng' if abs(all_stds.mean() - 1) < 0.05 else '❌ Không đúng'}")

**Expected Output:**

![img5-13](images/img_buoi5/img5-13.png)

<details>
<summary>👆 Click để xem đáp án Bài Tập 3</summary>

```python
all_means = X_train.mean()
print(f"Mean trung bình toàn X_train: {all_means.mean():.6f}")
# → ≈ 0.000000  ✅

all_stds = X_train.std()
print(f"Std trung bình toàn X_train : {all_stds.mean():.6f}")
# → ≈ 1.000000  ✅
```

**Nhận xét:** Kết quả mean ≈ 0 và std ≈ 1 xác nhận StandardScaler đã hoạt động đúng. Điều này quan trọng vì các mô hình như Ridge và Lasso rất nhạy cảm với scale của features.

</details>

---

> ✅ **Hoàn thành 3 bài tập Buổi 5!**
> 🚀 **Tiếp theo:** Buổi 6 — Train mô hình Linear Regression, Ridge, Lasso!

---

## 🏁 Tổng Kết Buổi 5

---

### ✅ Những Gì Bạn Đã Làm Được Hôm Nay

| # | Task | Kết quả |
|---|---|---|
| 1.1 | 📐 Area Features | +6 features (TotalSF, TotalPorch, TotalBath, OverallScore...) |
| 1.2 | 🕰️ Age Features | +6 features (HouseAge, IsNew, HasRemod...) |
| 1.3 | 📈 Transform | +11 features (log, sq, sqrt) |
| 2.0 | 🔢 Ordinal Encoding | 19 cột quality → số có thứ tự |
| 2.1 | 🔍 Correlation Analysis | Top 20 features + visualisation |
| 2.2 | 🗑️ Variance Filtering | Xoá features gần hằng số |
| 2.3 | ⚖️ StandardScaler | X_train, X_test chuẩn hoá |

---

### 📦 Biến Quan Trọng Để Dùng Ở Buổi 6

```python
X_train   → (~1458, ~215)  — features đã scale, sẵn sàng train
X_test    → (~1459, ~215)  — features đã scale, sẵn sàng predict
y         → (1458,)        — log1p(SalePrice) — target
scaler    → StandardScaler — dùng lại để inverse_transform nếu cần
selector  → VarianceThreshold — dùng lại nếu cần apply trên data mới
```

---

### 🔭 Buổi 6 Sẽ Làm Gì?

> **Model Training** — Huấn luyện Linear Regression, Ridge, Lasso và so sánh!

```
Models Buổi 6:
   LinearRegression   → baseline, không regularization
   Ridge (L2)         → giảm overfitting bằng penalty bình phương coeff
   Lasso (L1)         → vừa giảm overfitting vừa chọn feature (coeff → 0)
   ElasticNet (L1+L2) → kết hợp cả hai

Metrics:
   RMSE(log) → mục tiêu < 0.12
   R²        → mục tiêu > 0.90
   Cross-validation 5-fold
```

---

> 💡 **Ghi nhớ:** Feature Engineering là **kỹ năng sáng tạo** — không có công thức cố định. Người giỏi FE thường đạt kết quả tốt hơn người chỉ biết tune model!

> **Hẹn gặp lại ở Buổi 6 — Model Training! 🚀**